# Chat Prompt Quickstart

This notebook shows how a declarative chat prompt is forged into a reusable runtime prompt object. It uses both inline YAML and referenced message fragments.

What you will do:

- forge a prompt defined entirely inside one YAML file
- forge a second prompt built through `ref` composition
- inspect how placeholders and image inputs are preserved
- trigger a validation error with an intentionally malformed message

The first code cell only sets up imports and resolves the repository root so the example can run from the notebook folder.


In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (p / 'agentic_forge').exists()), NOTEBOOK_DIR)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CHAT_DIR = PROJECT_ROOT / 'examples' / 'chat_prompt'
MATERIALS_DIR = CHAT_DIR / 'materials'
STRUCT_ASSETS = PROJECT_ROOT / 'examples' / 'structured_output' / 'assets'

from agentic_forge import forge
from agentic_forge.chat_prompt import ForgedChatPrompt


## Step 1: Forge the inline prompt

`materials/chat_prompt.yaml` defines the full prompt inline. It combines:

- a `system` instruction
- a `placeholder` entry named `history`
- a multimodal `human` message with text and image inputs

After forging, the printed structure should make it clear that the YAML order becomes the runtime message order.


In [ ]:
prompt: ForgedChatPrompt = forge(MATERIALS_DIR / 'chat_prompt.yaml')
messages = prompt.format_messages(
    history=[],
    user_input='Give me a short caption for the two images.',
    image_path_1=str(STRUCT_ASSETS / 'sample_with_boxes.jpg'),
    image_path_2=str(STRUCT_ASSETS / 'table_image.jpg'),
)

for msg in prompt.compact_for_display(messages):
    print(msg)


## Step 2: Forge the composed prompt

`materials/chat_prompt_composed.yaml` mixes inline blocks and `ref` blocks. This is the reusable pattern used across Agentic Forge: small YAML fragments can be referenced, then selectively overridden where needed.


In [ ]:
composed_prompt: ForgedChatPrompt = forge(MATERIALS_DIR / 'chat_prompt_composed.yaml')
messages = composed_prompt.format_messages(
    history=[],
    user_input='Summarize what you see in the attached photo.',
    image_path=str(STRUCT_ASSETS / 'sample_with_boxes.jpg'),
)

for msg in composed_prompt.compact_for_display(messages):
    print(msg)


## Step 3: Inspect validation failures

`materials/malformed_chat_message.yaml` is wrong on purpose. The goal is to show that message materials are validated before the runtime prompt is created, so schema mistakes fail early and close to the YAML source.


In [ ]:
try:
    forge(MATERIALS_DIR / 'malformed_chat_message.yaml')
except Exception as exc:
    print(type(exc).__name__)
    print(exc)
